# Agentic AI Chatbot with Tool Use

This notebook creates a simple **Agentic AI chatbot** using **LM Studio** as the local LLM server.

Tools included:

- Normal AI chat
- Wikipedia search
- Safe arithmetic calculator
- PDF analyzer using FAISS vector search
- DOCX documentation generator

Before running, open **LM Studio**, load a model, and start the local server at:

`http://localhost:1234/v1/chat/completions`

In [ ]:
# Install required libraries
%pip install -q langchain langchain-community langchain-classic faiss-cpu pypdf wikipedia python-docx sentence-transformers

In [ ]:
# Imports and setup
import os
import re
import ast
import operator
import textwrap
import requests
import wikipedia
from docx import Document

from langchain_classic.memory import ConversationBufferMemory
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
conversation_history = []

conversation_stats = {
    "user_turns": 0,
    "wiki_searches": 0,
    "pdf_queries": 0,
    "arithmetic": 0,
    "ai_chats": 0
}

print("[System] Setup complete.")

In [ ]:
# LM Studio chat function

def lm_studio_chat(prompt, history=None, endpoint="http://localhost:1234/v1/chat/completions", model="local-model"):
    messages = []

    if history:
        for user_msg, ai_msg in history:
            messages.append({"role": "user", "content": user_msg})
            messages.append({"role": "assistant", "content": ai_msg})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "temperature": 0.3,
        "stream": False
    }

    try:
        response = requests.post(endpoint, json=payload, timeout=120)
        data = response.json()
    except requests.RequestException as e:
        return f"[LM Studio Error] Request failed. Make sure LM Studio server is running. Details: {e}"
    except ValueError:
        return f"[LM Studio Error] Non-JSON response: {response.text}"

    if response.status_code >= 400:
        return f"[LM Studio Error] HTTP {response.status_code}: {data}"

    try:
        return data["choices"][0]["message"]["content"]
    except Exception:
        return f"[LM Studio Error] Unexpected response: {data}"

print("[System] LM Studio function ready.")

In [ ]:
# Helper: print long text nicely

def print_wrapped(text, width=100):
    wrapper = textwrap.TextWrapper(width=width, replace_whitespace=False)
    lines = str(text).split("
")
    wrapped_text = [wrapper.fill(line) for line in lines]
    print("
".join(wrapped_text))

print("[System] Print helper ready.")

In [ ]:
# Tool 1: Wikipedia search

def wiki_search(query, max_chars=6000):
    query = query.strip()
    if not query:
        return "No search query provided."

    try:
        matches = wikipedia.search(query, results=5)
        if not matches:
            return "No Wikipedia page found."

        page = wikipedia.page(matches[0], auto_suggest=False)
        content = page.content[:max_chars]
        return f"Title: {page.title}
URL: {page.url}

{content}"

    except wikipedia.exceptions.DisambiguationError as e:
        return "Multiple results found. Try being more specific: " + ", ".join(e.options[:5])
    except wikipedia.exceptions.PageError:
        return "No Wikipedia page found."
    except Exception as e:
        return f"Wiki search error: {e}"

print("[System] Wikipedia tool ready.")

In [ ]:
# Tool 2: Safe arithmetic calculator

allowed_operators = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
    ast.USub: operator.neg,
}

def safe_eval_math(expression):
    def eval_node(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in allowed_operators:
            return allowed_operators[type(node.op)](eval_node(node.left), eval_node(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in allowed_operators:
            return allowed_operators[type(node.op)](eval_node(node.operand))
        raise ValueError("Only basic arithmetic is allowed.")

    expression = expression.replace("^", "**")
    tree = ast.parse(expression, mode="eval")
    return eval_node(tree.body)

print("[System] Arithmetic tool ready.")

In [ ]:
# Tool 3: PDF analyzer

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
pdf_vectordb = None

def initialize_pdf_system(pdf_folder="."):
    try:
        pdf_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith(".pdf")]
        if not pdf_files:
            return None, "No PDF file found in the current folder."

        target_pdf = os.path.join(pdf_folder, pdf_files[0])
        loader = PyPDFLoader(target_pdf)
        raw_docs = loader.load()

        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
        docs = splitter.split_documents(raw_docs)

        vectordb = FAISS.from_documents(docs, embeddings)
        return vectordb, f"Indexed PDF: {pdf_files[0]}"

    except Exception as e:
        return None, f"Failed to index PDF: {e}"

def ask_pdf(question):
    global pdf_vectordb

    if pdf_vectordb is None:
        pdf_vectordb, message = initialize_pdf_system()
        print(f"[PDF Analyzer] {message}")

    if pdf_vectordb is None:
        return "No PDF is available. Put a PDF file in the same folder as this notebook, then run again."

    search_results = pdf_vectordb.similarity_search(question, k=3)
    context = "

".join([doc.page_content for doc in search_results])

    prompt = f"""
You are analyzing a PDF document.
Use only the context below to answer the question.

Context:
{context}

Question: {question}

Answer clearly and concisely. If the answer is not in the context, say that it was not found in the PDF.
"""
    return lm_studio_chat(prompt)

print("[System] PDF analyzer ready.")

In [ ]:
# Tool 4: DOCX documentation creator

def create_docx(text, filename="output.docx"):
    doc = Document()
    doc.add_heading("Generated Documentation", level=1)

    for paragraph in str(text).split("
"):
        if paragraph.strip():
            doc.add_paragraph(paragraph.strip())

    doc.save(filename)
    return filename

print("[System] DOCX tool ready.")

In [ ]:
# Tool selector

def decide_tool(user_input):
    prompt = f"""
Choose the best tool for the user's message.
Available tools: chat, wiki, arithmetic, pdf.

Rules:
- Use arithmetic for math expressions or calculations.
- Use wiki for general factual research about people, places, history, science, etc.
- Use pdf if the user asks about a PDF, document, file, paper, or uploaded content.
- Use chat for normal conversation, explanations, coding help, and anything else.

User message: {user_input}

Reply using exactly this format:
TOOL: tool_name
KEYWORDS: important keywords only
"""

    response = lm_studio_chat(prompt)
    tool_match = re.search(r"TOOL:\s*(\w+)", response, re.IGNORECASE)
    keywords_match = re.search(r"KEYWORDS:\s*(.+)", response, re.IGNORECASE | re.DOTALL)

    tool = tool_match.group(1).lower() if tool_match else "chat"
    keywords = keywords_match.group(1).strip() if keywords_match else user_input

    if tool not in ["chat", "wiki", "arithmetic", "pdf"]:
        tool = "chat"

    return tool, keywords, response

print("[System] Tool selector ready.")

In [ ]:
# Main chatbot loop

print("Agentic AI Chatbot is ready.")
print("Type q to quit.
")

while True:
    q = input("You: [q to quit] ")
    print("
[User]")
    print(q)

    if q.strip().lower() == "q":
        summary = lm_studio_chat("Summarize this conversation briefly: " + str(conversation_history))
        print("
[Conversation Summary]")
        print_wrapped(summary)

        print("
--- Conversation Statistics ---")
        for k, v in conversation_stats.items():
            print(f"{k}: {v}")
        print("Goodbye!")
        break

    conversation_stats["user_turns"] += 1

    print("
[System] Deciding tool...")
    tool, keywords, raw_decision = decide_tool(q)
    print(f"[Tool Selected] {tool}")

    final_answer = ""

    if tool == "wiki":
        conversation_stats["wiki_searches"] += 1
        wiki_result = wiki_search(keywords)

        final_answer = lm_studio_chat(
            "Summarize this Wikipedia information clearly and briefly:

" + wiki_result
        )

        print("
[Wikipedia Summary]")
        print_wrapped(final_answer)

        doc_choice = input("
Create documentation in DOCX? (y/n): ")
        if doc_choice.lower().startswith("y"):
            safe_keyword = re.sub(r"[^A-Za-z0-9_-]+", "_", keywords).strip("_") or "wiki_output"
            doc_filename = f"{safe_keyword}.docx"
            create_docx(final_answer, filename=doc_filename)
            print(f"DOCX created: {doc_filename}")

    elif tool == "pdf":
        conversation_stats["pdf_queries"] += 1
        final_answer = ask_pdf(q)

        print("
[PDF Analyzer]")
        print_wrapped(final_answer)

    elif tool == "arithmetic":
        conversation_stats["arithmetic"] += 1
        try:
            expression = keywords if keywords else q
            result = safe_eval_math(expression)
            final_answer = f"The answer is: {result}"
        except Exception:
            final_answer = lm_studio_chat(
                "Solve this arithmetic problem and show the answer clearly: " + q
            )

        print("
[Arithmetic]")
        print_wrapped(final_answer)

    else:
        conversation_stats["ai_chats"] += 1
        final_answer = lm_studio_chat(q, history=conversation_history)

        print("
[AI]")
        print_wrapped(final_answer)

    conversation_history.append((q, final_answer))
    print("
" + "-" * 70 + "
")